# ARISTA spatiotemporal communication and 3D focus-anchor plot — CytoBridge API

**Objective.** Reproduce ARISTA attention-derived cell-type communication, dense growth/composition panels, and the five-slice focus-anchor 3D panel using only public `CytoBridge` APIs. Success means all per-cell tables, attention summaries, interactive/static figures, and a runtime manifest are regenerated without the vendored legacy source tree.


## Plan

1. Resolve a portable published checkpoint or a newly trained current checkpoint.
2. Simulate observed and intermediate distributions with piecewise spatial warp.
3. Compute attention interactions through `compute_timepoint_communications`.
4. Export dense observed/generated growth maps and cell-type composition through shared APIs.
5. Render the shared focus-anchor 3D API with reaEGC-specific styling.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path
import sys
import torch

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'downstream_helpers').exists():
    REPO_ROOT = REPO_ROOT.parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from downstream_helpers.arista_api import (
    AristaSpatiotemporalConfig,
    assert_package_only_runtime,
    run_arista_spatiotemporal_api,
)
from downstream_helpers.runner import display_html_outputs, display_image_outputs, display_svg_outputs

SEED = 42
DEVICE = os.environ.get('CYTOBRIDGE_DEVICE', 'cuda' if torch.cuda.is_available() else 'cpu')
SMOKE = os.environ.get('CYTOBRIDGE_SMOKE', '0') == '1'
MODEL_FORMAT = os.environ.get('ARISTA_MODEL_FORMAT', 'legacy')
ALIGNED_H5AD = os.environ.get('ARISTA_ALIGNED_H5AD') or None
MODEL_DIR = os.environ.get('ARISTA_MODEL_DIR') or None
DEVICE, SMOKE, MODEL_FORMAT


## Configuration

The biological choices (ARISTA timepoints and the reaEGC focus label) stay here. Simulation, classifier caching, attention aggregation, lineage, and Plotly geometry are package functions shared with other datasets. Set the three `ARISTA_*` environment variables for a current retrained model; otherwise the notebook uses portable published assets through the package compatibility loader.


In [ ]:
if MODEL_FORMAT == 'current' and (ALIGNED_H5AD is None or MODEL_DIR is None):
    raise ValueError('Current mode requires ARISTA_ALIGNED_H5AD and ARISTA_MODEL_DIR.')

config = AristaSpatiotemporalConfig(
    output_name='arista_spatiotemporal_3d_api' + ('_smoke' if SMOKE else ''),
    aligned_h5ad=ALIGNED_H5AD,
    model_dir=MODEL_DIR,
    model_format=MODEL_FORMAT,
    time_points=(0.0, 1.0) if SMOKE else (0.0, 1.0, 2.0, 3.0, 4.0),
    interp_time_points=(0.5,) if SMOKE else (0.5, 1.5, 2.5, 3.5),
    plot_3d_time_points=(0.0, 0.5, 1.0) if SMOKE else (0.0, 0.5, 1.0, 1.5, 2.0),
    n_samples=16 if SMOKE else 7668,
    classifier_epochs=2 if SMOKE else 1000,
    classifier_knn_neighbors=1,
    spatial_warp_to_observed_piecewise=True,
    split_sde_dt=0.1 if SMOKE else 0.01,
    random_seed=SEED,
    device=DEVICE,
    run_communication=True,
    run_3d=True,
)
config


In [ ]:
result = run_arista_spatiotemporal_api(config)
assert_package_only_runtime()
result


## Results

The 3D plot shows observed and generated spatial slices, attention-derived within-slice communication arrows, and cross-time lineage ribbons. The same interpolation run also writes the S13 dense growth grid and S14b cell-type composition table/stacked bar, avoiding a second simulation with different random state. The static PNG is convenient for quick review; use HTML to rotate and inspect anchors. Smoke mode validates API wiring only.


In [ ]:
display_svg_outputs([result.growth_figure, result.composition_figure])
display_image_outputs([result.output_dir / 'spatiotemporal_3d.png'])
display_html_outputs([result.spatiotemporal_html], height=900)
print(result.manifest_path.read_text(encoding='utf-8'))


## Next checks

- Compare current-model and published-checkpoint communication matrices at the same timepoints.
- Inspect reaEGC outgoing edges before and after the wound-proximal spatial restriction.
- Use the manifest to confirm KNN=1, piecewise warp, package path, and model stages before manuscript figure replacement.
